# Phase 1 on free Google Colab

Runs the MedHallu baseline and one end-to-end check of the full system. Everything is free.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU`.
2. Open the **Secrets** panel (key icon on the left) and add `NCBI_EMAIL`. Switch on "Notebook access" for it.
3. Run the cells in order. The last cell downloads all results as a zip.

All three verifiers (Llama 3.1 8B, Mistral 7B and Phi-4-mini) run on Colab's GPU through Ollama, so no paid or rate-limited API is needed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install Ollama and download the three local models

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull llama3.1:8b
!ollama pull phi4-mini
!ollama pull mistral:7b
!ollama list

## 2. Get the code

If the Colab session restarts, run this cell again: it pulls the latest code instead of cloning twice.

In [ ]:
import os
REPO_URL = "https://github.com/khan-shayan9/Agentic-AI-Project.git"
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo
!pip install -q -e .

## 3. Load your keys from Colab Secrets

`NCBI_EMAIL` is needed for PubMed (step 7). `NCBI_API_KEY`, `HF_TOKEN` and `MISTRAL_API_KEY` (only if you want the Mistral API instead of the local model) are optional.

In [ ]:
import os
try:
    from google.colab import userdata
    lines = []
    for name in ["MISTRAL_API_KEY", "NCBI_EMAIL", "NCBI_API_KEY", "HF_TOKEN"]:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            lines.append(f"{name}={value}")
            os.environ[name] = value
    open(".env", "w").write("\n".join(lines) + "\n")
    print("Loaded:", [l.split("=")[0] for l in lines])
except ImportError:
    print("Not running on Colab; use a .env file instead.")

## 4. Prepare the data (download MedHallu, attach PubMed IDs, make the splits)

In [ ]:
!python -m medverify prepare

## 5. Quick check on the 20-row dev split

In [ ]:
!python -m medverify baseline --model llama --split dev --name llama-dev-none

## 6. Baseline on the test split (600 answers each)

- **without knowledge** = MedHallu's main setting (B0)
- **with source abstract** = MedHallu's knowledge setting, the baseline the instructor asked for (B1)

MedHallu reported Llama-3.1-8B-Instruct at F1 0.522 without and 0.797 with knowledge. Ours should be in the same area; differences come from the smaller test set and Ollama's 4-bit model.

In [ ]:
!python -m medverify baseline --model llama --split test --knowledge none --name llama-test-none

In [ ]:
!python -m medverify baseline --model llama --split test --knowledge source --name llama-test-source

Optional: the same baseline with the "not sure" option (B2).

In [ ]:
!python -m medverify baseline --model llama --split test --knowledge none --unsure --name llama-test-none-unsure

## 7. Full system on the dev split (end-to-end check)

Evidence agent (PubMed, source blocked) + three verifiers (Llama 3.1 8B, Mistral 7B, Phi-4-mini, all local) + vote with abstention, on 40 dev answers. Verifiers answer with a word (FACTUAL, HALLUCINATED or UNSURE).


In [ ]:
!python -m medverify pipeline --split dev --evidence pubmed --name sysA-dev && python -m medverify evaluate results/runs/sysA-dev && python -m medverify overlap results/runs/sysA-dev


Check how each verifier answered (counts of 0 = factual, 1 = hallucinated, 2 = unsure, None = unreadable).

In [ ]:
import json, collections
rows = [json.loads(l) for l in open("results/runs/sysA-dev/states.jsonl")]
for name in ["llama", "mistral-local", "phi"]:
    vs = [v for r in rows for v in r["verdicts"] if v["system"] == f"verifier:{name}"]
    print(name, dict(collections.Counter(str(v["verdict"]) for v in vs)))


## 8. Numbers for the proposal's results table

In [ ]:
import json, pathlib
for run in ["llama-test-none", "llama-test-source", "sysA-dev"]:
    path = pathlib.Path("results/runs") / run / "metrics.json"
    if not path.exists():
        print(f"{run}: not run yet")
        continue
    m = json.loads(path.read_text())
    o, ci = m["overall"], m["overall"].get("f1_ci95")
    ci_text = f" ({ci[0]:.3f}-{ci[1]:.3f})" if ci else ""
    levels = "  ".join(f"{lvl} F1 {m[lvl]['f1']:.3f}" for lvl in ("easy", "medium", "hard") if lvl in m)
    print(f"{run}: precision {o['precision']:.3f}  recall {o['recall']:.3f}  F1 {o['f1']:.3f}{ci_text}  coverage {o['coverage']:.3f}  {levels}")

## 9. Download the results

In [ ]:
!zip -qr results.zip results data/processed
from google.colab import files
files.download("results.zip")